# Post-Set-1 Brier Score

After set 1 completes, we know:
- Who won set 1 (strong signal)
- Per-set stats for both players (fresh observations)

We use set 1 stats as the BetaModel prior, start the simulation from the known set score (1-0 or 0-1), and measure how well we predict the final match winner.

Comparison baselines:
- **Random**: 0.2500
- **Kalshi opening**: 0.2293
- **MC + YTD (pre-match)**: 0.2216
- **Naive (set 1 winner always wins)**: computed below

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
class BetaModel:
    def __init__(self, career_rate, prior_strength, lam=0.95, warmup=None):
        self.alpha_prior = career_rate * prior_strength
        self.beta_prior  = (1.0 - career_rate) * prior_strength
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.lam    = lam
        self.warmup = warmup if warmup is not None else int(1 / (1 - lam))
        self.n_obs  = 0

    def get_p(self):
        if self.n_obs < self.warmup:
            return self.alpha_prior / (self.alpha_prior + self.beta_prior)
        total = (self.alpha_prior + self.beta_prior +
                 self.alpha_match + self.beta_match)
        return (self.alpha_prior + self.alpha_match) / total

    def update(self, won):
        self.alpha_match *= self.lam
        self.beta_match  *= self.lam
        if won: self.alpha_match += 1.0
        else:   self.beta_match  += 1.0
        self.n_obs += 1

    def reset(self):
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.n_obs       = 0


class Player:
    def __init__(self, name, stats, prior_strength=200, lam=0.95):
        self.name  = name
        self.stats = stats
        self.serve_first_model   = BetaModel(stats['win_first'],    prior_strength, lam)
        self.serve_second_model  = BetaModel(stats['win_second'],   prior_strength, lam)
        self.return_first_model  = BetaModel(stats['return_first'], prior_strength, lam)
        self.return_second_model = BetaModel(stats['return_second'],prior_strength, lam)
        self.bp_save_model = BetaModel(
            career_rate=stats['bp_save_rate'], prior_strength=stats['bp_save_faced'],
            lam=lam, warmup=20)
        self.bp_convert_model = BetaModel(
            career_rate=stats['bp_convert_rate'], prior_strength=stats['bp_convert_opps'],
            lam=lam, warmup=20)

    def reset(self):
        for m in [self.serve_first_model, self.serve_second_model,
                  self.return_first_model, self.return_second_model,
                  self.bp_save_model, self.bp_convert_model]:
            m.reset()


def is_break_point(server_pts, receiver_pts):
    if receiver_pts == 3 and server_pts < 3: return True
    if receiver_pts >= 4 and server_pts >= 3 and receiver_pts == server_pts + 1: return True
    return False


def sim_point(server, receiver, server_pts, receiver_pts):
    bp = is_break_point(server_pts, receiver_pts)
    first_in = server.stats['first_in']
    if bp:
        p_win = (server.bp_save_model.get_p() + (1.0 - receiver.bp_convert_model.get_p())) / 2.0
        server_won = np.random.random() < p_win
        server.bp_save_model.update(server_won)
        receiver.bp_convert_model.update(not server_won)
        if np.random.random() < first_in:
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)
    else:
        p_win_1st = (server.serve_first_model.get_p() + (1.0 - receiver.return_first_model.get_p())) / 2.0
        p_win_2nd = (server.serve_second_model.get_p() + (1.0 - receiver.return_second_model.get_p())) / 2.0
        if np.random.random() < first_in:
            server_won = np.random.random() < p_win_1st
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server_won = np.random.random() < p_win_2nd
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)
    return server_won


def sim_game(server, receiver):
    score = [0, 0]
    while True:
        if sim_point(server, receiver, score[0], score[1]): score[0] += 1
        else: score[1] += 1
        if score[0] >= 4 and score[0] - score[1] >= 2: return True
        if score[1] >= 4 and score[1] - score[0] >= 2: return False


def sim_tiebreak(p1, p2, p1_serves_first):
    score = [0, 0]
    point_count = 0
    while True:
        if point_count == 0: p1_serves = p1_serves_first
        else: p1_serves = p1_serves_first == (point_count % 2 == 0)
        server, receiver = (p1, p2) if p1_serves else (p2, p1)
        server_won = sim_point(server, receiver, 0, 0)
        p1_won = server_won if p1_serves else not server_won
        score[0 if p1_won else 1] += 1
        point_count += 1
        if score[0] >= 7 and score[0] - score[1] >= 2: return True
        if score[1] >= 7 and score[1] - score[0] >= 2: return False


def sim_set(p1, p2, p1_serves_first):
    games = [0, 0]
    p1_serving = p1_serves_first
    while True:
        server, receiver = (p1, p2) if p1_serving else (p2, p1)
        server_won = sim_game(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won
        games[0 if p1_won_game else 1] += 1
        p1_serving = not p1_serving
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            games[0 if p1_won_tb else 1] += 1
            return games[0] > games[1], p1_serving
        if games[0] >= 6 and games[0] - games[1] >= 2: return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2: return False, p1_serving


def sim_match(p1, p2, p1_serves_first=True, best_of=3, starting_sets=(0, 0)):
    p1.reset()
    p2.reset()
    sets_needed = best_of // 2 + 1
    sets = list(starting_sets)
    p1_serving = p1_serves_first
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1, p2, p1_serving)
        sets[0 if p1_won_set else 1] += 1
    return sets[0] > sets[1]


def run_simulation(p1, p2, N=5000, best_of=3, starting_sets=(0, 0)):
    wins = sum(sim_match(p1, p2, best_of=best_of, starting_sets=starting_sets) for _ in range(N))
    return wins / N

In [3]:
DB_PATH = 'tennis.db'
N_SIMS  = 5000
PRIOR_STRENGTH = 200
LAM = 0.95

conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    m.id            AS match_id,
    m.winner_player_id,
    m.number_of_sets,
    mp1.player_id   AS p1_id,
    mp2.player_id   AS p2_id,
    s1p1.set_score  AS p1_set1_score,
    s1p2.set_score  AS p2_set1_score,
    s1p1.first_serve_pct        AS p1_first_in,
    s1p1.first_serve_won_pct    AS p1_win_first,
    s1p1.second_serve_won_pct   AS p1_win_second,
    s1p1.first_return_won_pct   AS p1_return_first,
    s1p1.second_return_won_pct  AS p1_return_second,
    s1p1.bp_saved_pct           AS p1_bp_save_rate,
    s1p1.bp_faced               AS p1_bp_save_faced,
    s1p1.bp_converted_pct       AS p1_bp_convert_rate,
    s1p1.bp_opportunities       AS p1_bp_convert_opps,
    s1p2.first_serve_pct        AS p2_first_in,
    s1p2.first_serve_won_pct    AS p2_win_first,
    s1p2.second_serve_won_pct   AS p2_win_second,
    s1p2.first_return_won_pct   AS p2_return_first,
    s1p2.second_return_won_pct  AS p2_return_second,
    s1p2.bp_saved_pct           AS p2_bp_save_rate,
    s1p2.bp_faced               AS p2_bp_save_faced,
    s1p2.bp_converted_pct       AS p2_bp_convert_rate,
    s1p2.bp_opportunities       AS p2_bp_convert_opps
FROM matches m
JOIN match_players mp1  ON mp1.match_id  = m.id AND mp1.is_player1 = 1
JOIN match_players mp2  ON mp2.match_id  = m.id AND mp2.is_player1 = 0
JOIN set_stats s1p1 ON s1p1.match_id = m.id AND s1p1.player_id = mp1.player_id AND s1p1.set_number = 1
JOIN set_stats s1p2 ON s1p2.match_id = m.id AND s1p2.player_id = mp2.player_id AND s1p2.set_number = 1
WHERE m.winner_player_id IS NOT NULL
"""

df = pd.read_sql_query(query, conn)
conn.close()
print(f'Matches loaded: {len(df):,}')

Matches loaded: 1,235


In [4]:
stat_cols = [
    'p1_first_in','p1_win_first','p1_win_second','p1_return_first','p1_return_second',
    'p1_bp_save_rate','p1_bp_save_faced','p1_bp_convert_rate','p1_bp_convert_opps',
    'p2_first_in','p2_win_first','p2_win_second','p2_return_first','p2_return_second',
    'p2_bp_save_rate','p2_bp_save_faced','p2_bp_convert_rate','p2_bp_convert_opps',
]
before = len(df)
df = df.dropna(subset=stat_cols)
df = df[(df['p1_bp_save_faced'] > 0) & (df['p1_bp_convert_opps'] > 0) &
        (df['p2_bp_save_faced'] > 0) & (df['p2_bp_convert_opps'] > 0)]
print(f'Dropped {before - len(df):,} rows, {len(df):,} remain')

df['p1_won_set1'] = df['p1_set1_score'].astype(int) > df['p2_set1_score'].astype(int)
df['starting_sets_p1'] = df['p1_won_set1'].map({True: 1, False: 0})
df['starting_sets_p2'] = df['p1_won_set1'].map({True: 0, False: 1})
df['outcome'] = (df['winner_player_id'] == df['p1_id']).astype(int)

print(f'P1 won set 1: {df["p1_won_set1"].sum():,} ({df["p1_won_set1"].mean()*100:.1f}%)')

Dropped 503 rows, 732 remain
P1 won set 1: 342 (46.7%)


In [5]:
naive_pred = df['starting_sets_p1'].values
naive_brier = float(np.mean((naive_pred - df['outcome'].values) ** 2))
naive_accuracy = float(np.mean(naive_pred == df['outcome'].values))
print(f'Naive baseline (set 1 winner always wins):')
print(f'  Brier score : {naive_brier:.4f}')
print(f'  Accuracy    : {naive_accuracy*100:.1f}%')

Naive baseline (set 1 winner always wins):
  Brier score : 0.1762
  Accuracy    : 82.4%


In [6]:
def make_stats(row, prefix):
    return {
        'first_in':        row[f'{prefix}first_in']        / 100,
        'win_first':       row[f'{prefix}win_first']       / 100,
        'win_second':      row[f'{prefix}win_second']      / 100,
        'return_first':    row[f'{prefix}return_first']    / 100,
        'return_second':   row[f'{prefix}return_second']   / 100,
        'bp_save_rate':    row[f'{prefix}bp_save_rate']    / 100,
        'bp_save_faced':   max(1, int(row[f'{prefix}bp_save_faced'])),
        'bp_convert_rate': row[f'{prefix}bp_convert_rate'] / 100,
        'bp_convert_opps': max(1, int(row[f'{prefix}bp_convert_opps'])),
    }


results = []
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Simulating'):
    try:
        s1 = make_stats(row, 'p1_')
        s2 = make_stats(row, 'p2_')
        p1 = Player('p1', s1, prior_strength=PRIOR_STRENGTH, lam=LAM)
        p2 = Player('p2', s2, prior_strength=PRIOR_STRENGTH, lam=LAM)
        best_of = int(row['number_of_sets']) if row['number_of_sets'] else 3
        starting = (int(row['starting_sets_p1']), int(row['starting_sets_p2']))
        p1_prob = run_simulation(p1, p2, N=N_SIMS, best_of=best_of, starting_sets=starting)
        results.append({
            'match_id':    row['match_id'],
            'p1_won_set1': row['p1_won_set1'],
            'p1_prob':     p1_prob,
            'outcome':     row['outcome'],
            'brier':       (p1_prob - row['outcome']) ** 2,
        })
    except Exception as e:
        skipped += 1

res = pd.DataFrame(results)
print(f'Simulated: {len(res):,}  |  Skipped: {skipped}')

Simulating: 100%|██████████| 732/732 [06:17<00:00,  1.94it/s]

Simulated: 732  |  Skipped: 0


In [7]:
brier = res['brier'].mean()
print('=' * 52)
print(f'Post-set-1 MC Brier score : {brier:.4f}')
print(f'Skill score               : {1 - brier/0.25:.4f}')
print()
print('Comparison:')
print(f'  Random baseline          : 0.2500  (skill=0.000)')
print(f'  Kalshi opening price     : 0.2293  (skill=0.083)')
print(f'  MC + YTD pre-match       : 0.2216  (skill=0.114)')
print(f'  Naive (set1 winner wins) : {naive_brier:.4f}  (skill={1-naive_brier/0.25:.3f})')
print(f'  MC + set1 stats (this)   : {brier:.4f}  (skill={1-brier/0.25:.3f})')
print('=' * 52)
print()
print('By set 1 result:')
print(res.groupby('p1_won_set1')['brier'].agg(['mean','count']).round(4))

Post-set-1 MC Brier score : 0.1625
Skill score               : 0.3499

Comparison:
  Random baseline          : 0.2500  (skill=0.000)
  Kalshi opening price     : 0.2293  (skill=0.083)
  MC + YTD pre-match       : 0.2216  (skill=0.114)
  Naive (set1 winner wins) : 0.1762  (skill=0.295)
  MC + set1 stats (this)   : 0.1625  (skill=0.350)

By set 1 result:
               mean  count
p1_won_set1               
False        0.1678    390
True         0.1565    342


In [8]:
res.to_csv('set1_brier_results.csv', index=False)
print('Saved to set1_brier_results.csv')

Saved to set1_brier_results.csv


## Approach 2: No break point models

All points — including break point situations — use the normal serve/return stats.
Break point counts from set 1 are sparse (2–5 observations), so this tests whether
removing the noisy bp prior helps or hurts.

Only `sim_game` changes: it passes a fixed score of (0, 0) so `is_break_point` always
returns False. Everything else (tiebreak, set, match) is identical.

In [9]:
def sim_game_no_bp(server, receiver):
    """Same as sim_game but never triggers break point logic."""
    score = [0, 0]
    while True:
        # Pass (0, 0) so is_break_point always returns False
        if sim_point(server, receiver, 0, 0): score[0] += 1
        else: score[1] += 1
        if score[0] >= 4 and score[0] - score[1] >= 2: return True
        if score[1] >= 4 and score[1] - score[0] >= 2: return False


def sim_set_no_bp(p1, p2, p1_serves_first):
    games = [0, 0]
    p1_serving = p1_serves_first
    while True:
        server, receiver = (p1, p2) if p1_serving else (p2, p1)
        server_won = sim_game_no_bp(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won
        games[0 if p1_won_game else 1] += 1
        p1_serving = not p1_serving
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            games[0 if p1_won_tb else 1] += 1
            return games[0] > games[1], p1_serving
        if games[0] >= 6 and games[0] - games[1] >= 2: return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2: return False, p1_serving


def sim_match_no_bp(p1, p2, p1_serves_first=True, best_of=3, starting_sets=(0, 0)):
    p1.reset()
    p2.reset()
    sets_needed = best_of // 2 + 1
    sets = list(starting_sets)
    p1_serving = p1_serves_first
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set_no_bp(p1, p2, p1_serving)
        sets[0 if p1_won_set else 1] += 1
    return sets[0] > sets[1]


def run_simulation_no_bp(p1, p2, N=5000, best_of=3, starting_sets=(0, 0)):
    wins = sum(sim_match_no_bp(p1, p2, best_of=best_of, starting_sets=starting_sets) for _ in range(N))
    return wins / N


results_no_bp = []
skipped_no_bp = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Simulating (no BP)'):
    try:
        s1 = make_stats(row, 'p1_')
        s2 = make_stats(row, 'p2_')
        p1 = Player('p1', s1, prior_strength=PRIOR_STRENGTH, lam=LAM)
        p2 = Player('p2', s2, prior_strength=PRIOR_STRENGTH, lam=LAM)
        best_of = int(row['number_of_sets']) if row['number_of_sets'] else 3
        starting = (int(row['starting_sets_p1']), int(row['starting_sets_p2']))
        p1_prob = run_simulation_no_bp(p1, p2, N=N_SIMS, best_of=best_of, starting_sets=starting)
        results_no_bp.append({
            'match_id':    row['match_id'],
            'p1_won_set1': row['p1_won_set1'],
            'p1_prob':     p1_prob,
            'outcome':     row['outcome'],
            'brier':       (p1_prob - row['outcome']) ** 2,
        })
    except Exception as e:
        skipped_no_bp += 1

res_no_bp = pd.DataFrame(results_no_bp)
print(f'Simulated: {len(res_no_bp):,}  |  Skipped: {skipped_no_bp}')

Simulating (no BP): 100%|██████████| 732/732 [04:59<00:00,  2.45it/s]

Simulated: 732  |  Skipped: 0


In [10]:
# ── Final comparison ──────────────────────────────────────────────────────────
brier_bp    = res['brier'].mean()
brier_no_bp = res_no_bp['brier'].mean()
random_brier = 0.25

def skill(b): return 1 - b / random_brier

print('─' * 60)
print(f'{"Model":<32}  {"Brier":>7}  {"Skill":>7}')
print('─' * 60)
print(f'{"Random baseline":<32}  {0.2500:.4f}  {skill(0.2500):>7.4f}')
print(f'{"Kalshi opening price":<32}  {0.2293:.4f}  {skill(0.2293):>7.4f}')
print(f'{"MC + YTD pre-match (clean)":<32}  {0.2216:.4f}  {skill(0.2216):>7.4f}')
print(f'{"Naive (set 1 winner wins)":<32}  {naive_brier:.4f}  {skill(naive_brier):>7.4f}')
print(f'{"MC post-set-1 (with BP)":<32}  {brier_bp:.4f}  {skill(brier_bp):>7.4f}')
print(f'{"MC post-set-1 (no BP)":<32}  {brier_no_bp:.4f}  {skill(brier_no_bp):>7.4f}')
print('─' * 60)
print(f'\nMatches in set-1 analysis : {len(res):,}')
print(f'BP vs no-BP difference    : {brier_no_bp - brier_bp:+.4f} Brier pts  (+ = no-BP is worse)')

res_no_bp.to_csv('set1_brier_no_bp_results.csv', index=False)
print('\nSaved → set1_brier_no_bp_results.csv')

────────────────────────────────────────────────────────────
Model                               Brier    Skill
────────────────────────────────────────────────────────────
Random baseline                   0.2500   0.0000
Kalshi opening price              0.2293   0.0828
MC + YTD pre-match (clean)        0.2216   0.1136
Naive (set 1 winner wins)         0.1762   0.2951
MC post-set-1 (with BP)           0.1625   0.3499
MC post-set-1 (no BP)             0.1591   0.3638
────────────────────────────────────────────────────────────

Matches in set-1 analysis : 732
BP vs no-BP difference    : -0.0035 Brier pts  (+ = no-BP is worse)

Saved → set1_brier_no_bp_results.csv
